# Genomic prediction of infliximab response in IBD — discovery pipeline

Trains an $L_1$-penalised logistic regression on baseline mucosal transcriptomic profiles from
[GSE16879](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE16879) to predict primary response
to anti-TNF therapy, and estimates generalisation performance by nested cross-validation.

Writes `infliximab_lasso_model.pkl` and `infliximab_coefficients.csv`, both consumed by
`04_multi_cohort_validation_final.ipynb`.

## 1. Data ingestion

Downloads the GSE16879 series matrix from NCBI GEO if it is not already cached locally.

In [29]:
import pandas as pd
import numpy as np
import os
import urllib.request
import gzip

data_url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE16nnn/GSE16879/matrix/GSE16879_series_matrix.txt.gz"
dest_path = os.path.join("data", "GSE16879_series_matrix.txt.gz")
os.makedirs("data", exist_ok=True)

if not os.path.exists(dest_path):
    print("Downloading dataset from NCBI GEO... This might take a minute.")
    urllib.request.urlretrieve(data_url, dest_path)
    print(f"Download complete! File saved to: {dest_path}")
else:
    print("Dataset already exists locally in the 'data' folder.")

file_path = dest_path

Dataset already exists locally in the 'data' folder.


## 2. Clinical metadata parsing

GEO series matrix files prefix every metadata line with `!`. Sample characteristics are spread across
repeated `!Sample_characteristics_ch1` lines, so duplicate keys are suffixed with a counter to keep
them distinct.

In [30]:
metadata_dict = {}

with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        # stop reading once the numeric expression block begins
        if not line.startswith("!"):
            if line.strip() and not line.startswith('"ID_REF"'):
                break

        if line.startswith("!Sample_"):
            parts = line.strip().split("\t")
            base_key = parts[0].replace('!Sample_', '').strip()
            values = [v.replace('"', '').strip() for v in parts[1:]]

            # de-duplicate repeated keys: characteristics_ch1, characteristics_ch1_1, ...
            final_key = base_key
            counter = 1
            while final_key in metadata_dict:
                final_key = f"{base_key}_{counter}"
                counter += 1

            metadata_dict[final_key] = values

df_clinical = pd.DataFrame.from_dict(metadata_dict)
print(f"Parsed {df_clinical.shape[0]} samples, {df_clinical.shape[1]} metadata fields")

Parsed 133 samples, 35 metadata fields


The four `characteristics_ch1*` fields carry tissue, disease, response, and treatment timepoint.

In [31]:
df_clinical[['characteristics_ch1', 'characteristics_ch1_1',
             'characteristics_ch1_2', 'characteristics_ch1_3']].tail(5)

,characteristics_ch1,characteristics_ch1_1,characteristics_ch1_2,characteristics_ch1_3
128,tissue: Ileum,disease: CD,response to infliximab: No,before or after first infliximab treatment: Af...
129,tissue: Ileum,disease: CD,response to infliximab: No,before or after first infliximab treatment: Be...
130,tissue: Ileum,disease: CD,response to infliximab: No,before or after first infliximab treatment: Af...
131,tissue: Ileum,disease: CD,response to infliximab: No,before or after first infliximab treatment: Be...
132,tissue: Ileum,disease: CD,response to infliximab: No,before or after first infliximab treatment: Af...


## 3. Cohort filtering

Two filters are applied.

**Healthy controls** carry `response to infliximab: Not applicable` and are removed.

**Post-treatment samples.** GSE16879 profiles each patient both before and after their first
infliximab infusion. Response is a patient-level label, so both biopsies from a given patient carry
the same `Yes`/`No` value. Retaining both would place the same patient on either side of a train/test
split with an identical label attached, allowing the model to identify individual patients from their
expression profile rather than learn response biology. Only baseline (pre-treatment) profiles are
retained, giving one row per patient.

In [32]:
# remove healthy controls
df_ibd = df_clinical[df_clinical['characteristics_ch1_2'] != "response to infliximab: Not applicable"].copy()

df_ibd = df_ibd.rename(columns={
    'characteristics_ch1':   'tissue',
    'characteristics_ch1_1': 'disease',
    'characteristics_ch1_2': 'response to infliximab',
    'characteristics_ch1_3': 'before or after first infliximab treatment',
})

# strip the "field: " prefix GEO embeds in every value
for col, prefix in [
    ('tissue', 'tissue: '),
    ('disease', 'disease: '),
    ('response to infliximab', 'response to infliximab: '),
    ('before or after first infliximab treatment', 'before or after first infliximab treatment: '),
]:
    df_ibd[col] = df_ibd[col].str.replace(prefix, "", regex=False)

print(f"IBD samples after removing controls: {df_ibd.shape[0]}")

IBD samples after removing controls: 121


Sample composition by treatment timepoint:

In [33]:
print(df_ibd['before or after first infliximab treatment'].value_counts())

before or after first infliximab treatment
Before first infliximab treatment    61
After first infliximab treatment     60
Name: count, dtype: int64


The 121 IBD samples comprise 61 pre-treatment and 60 post-treatment biopsies drawn from the same patients. Filtering to baseline leaves 61 independent patients.

In [34]:
# retain baseline (pre-treatment) profiles only — one row per patient
df_ibd = df_ibd[df_ibd["before or after first infliximab treatment"] == 'Before first infliximab treatment']

print(f"Baseline cohort: {df_ibd.shape[0]} patients")
print(df_ibd['response to infliximab'].value_counts())

Baseline cohort: 61 patients
response to infliximab
No     33
Yes    28
Name: count, dtype: int64


**Majority-class baseline: 54.1%.** The best constant predictor (`No` for every patient) is correct
on 33 of 61. Accuracy figures below are measured against this value, not against 50%.

## 4. Expression matrix alignment

Loads the 54,675-probe matrix, transposes it so that rows are patients, and restricts it to the
baseline cohort. `comment='!'` skips the metadata header parsed in section 2.

In [35]:
df_genes = pd.read_csv(
    file_path,
    sep='\t',
    compression='gzip',
    comment='!',        # skip the clinical header block
    index_col=0,        # probe IDs become row names
)

df_genes = df_genes.T                       # rows = patients, columns = probes
df_ibd = df_ibd.set_index('geo_accession')  # align on GEO sample accession
df_genes = df_genes.loc[df_ibd.index]       # restrict to the baseline cohort

print("Clinical shape:", df_ibd.shape)
print("Gene shape:", df_genes.shape)

Clinical shape: (61, 34)
Gene shape: (61, 54675)


## 5. Nested cross-validation

Three elements of the design are load-bearing.

**Nesting.** An inner 3-fold `GridSearchCV` tunes `C`; an outer 5-fold loop scores each tuned model
on data the inner loop never saw. Passing `grid_search` to `cross_val_score` as the estimator is what
makes the procedure nested — the full inner search re-runs from scratch within each outer fold. A
single CV loop that both selected `C` and reported the resulting score would leak the test fold into
the tuning decision and bias the estimate upward.

**Per-fold standardisation.** `StandardScaler` sits inside the `Pipeline` and is therefore re-fitted
on each training fold rather than once across all 61 patients, which would leak test-fold means and
variances. Scaling is material for $L_1$ specifically: the penalty is the sum of absolute
coefficients, and a low-variance probe requires a larger coefficient — and so incurs a larger penalty
— to exert the same effect on the prediction. Unstandardised $L_1$ is consequently biased toward
high-variance probes irrespective of their biological relevance.

**Stratification.** With approximately 12 patients per outer test fold, an unstratified split could
produce a fold composed almost entirely of one class. `StratifiedKFold` preserves the 33/28 ratio
within every fold.

Nested cross-validation yields a performance estimate rather than a model; the tuned models it
constructs are discarded. The deployable model is fitted separately in section 6.

In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

y = df_ibd['response to infliximab']

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(penalty='l1', solver='liblinear', random_state=2)),
])

param_grid = {'clf__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]}

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=2)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

grid_search = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='accuracy')
nested_scores = cross_val_score(grid_search, df_genes, y, cv=outer_cv, scoring='accuracy')

print(nested_scores)
print(f"Nested CV accuracy: {nested_scores.mean()*100:.2f}% (+/- {nested_scores.std()*100:.2f}%)")

[0.61538462 0.66666667 0.58333333 0.75       0.83333333]
Nested CV accuracy: 68.97% (+/- 9.13%)


**68.97% ± 9.13%**, against a majority-class baseline of 54.1%. Fold-to-fold variance is wide by
construction at approximately 12 patients per test fold; the mean is the estimate, and individual
folds should not be interpreted.

## 6. Final model fit

Refits on all 61 patients with no outer holdout. This is the artefact evaluated against the external
cohort.

In [37]:
final_search = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='accuracy')
final_search.fit(df_genes, y)
final_model = final_search.best_estimator_

print(f"Chosen C: {final_search.best_params_['clf__C']}")

Chosen C: 100


`C = 100` is interior to the search grid, so the selected value is a genuine optimum rather than one clipped at a grid boundary.

## 7. Surviving features

The $L_1$ penalty drives most coefficients to exactly zero; the survivors constitute the selected feature set.

In [38]:
df_results = pd.DataFrame({
    'probe':  df_genes.columns,
    'weight': final_model.named_steps['clf'].coef_[0],
})
df_results = df_results[df_results['weight'] != 0.0].copy()

print(f"Surviving probes: {len(df_results)}  ({(1 - len(df_results)/df_genes.shape[1])*100:.1f}% eliminated)")

Surviving probes: 704  (98.7% eliminated)


## 8. Selection stability

Tests whether the selected feature set is reproducible under resampling. The model is refit on each
outer training fold and the selection frequency of every probe recorded. A probe carrying genuine
signal should be selected irrespective of which patients are held out.

In [39]:
from collections import Counter

survivors = Counter()
for train_idx, _ in outer_cv.split(df_genes, y):
    gs = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='accuracy')
    gs.fit(df_genes.iloc[train_idx], y.iloc[train_idx])
    coefs = gs.best_estimator_.named_steps['clf'].coef_[0]
    survivors.update(df_genes.columns[coefs != 0])

print(survivors.most_common(15))

[('1552726_at', 5), ('1568787_at', 5), ('203349_s_at', 5), ('225807_at', 5), ('1569830_at', 4), ('204475_at', 4), ('206172_at', 4), ('219793_at', 4), ('229672_at', 4), ('235364_at', 4), ('239384_at', 4), ('241164_at', 4), ('243962_at', 4), ('1564253_at', 4), ('205931_s_at', 4)]


In [40]:
print("Probes selected in all 5 folds:", sum(1 for v in survivors.values() if v == 5))
print("Union of probes selected across folds:", len(survivors))
print("Selection frequency of 216242_x_at (top weight in the final fit):", survivors['216242_x_at'], "of 5")

Probes selected in all 5 folds: 4
Union of probes selected across folds: 3322
Selection frequency of 216242_x_at (top weight in the final fit): 3 of 5


Each fold selects approximately 704 probes. Across five folds these union to **3,322 distinct
probes** against an intersection of **4** (`1552726_at`, `1568787_at`, `203349_s_at`, `225807_at`):
the folds select near-disjoint feature sets from the same 61 patients.

`216242_x_at`, the highest-weighted probe in the final fit, is selected in 3 of 5 folds. Coefficient
magnitude and selection frequency are largely decoupled.

At n=61 against 54,675 features the selected signature is not reproducible under resampling, and no
individual probe from this model supports a biomarker claim.

## 9. Export

The full coefficient table is written with each probe's fold-selection count, sorted by absolute
weight. No top-N biomarker list is produced: on the evidence of section 8 such a list would reflect a
single arbitrary partition of the data.

In [41]:
import joblib

df_results['folds_selected'] = df_results['probe'].map(survivors).fillna(0).astype(int)
df_results = df_results.sort_values('weight', key=abs, ascending=False)
df_results.to_csv("infliximab_coefficients.csv", index=False)

joblib.dump(final_model, "infliximab_lasso_model.pkl")
print(f"Exported {len(df_results)} coefficients and the fitted pipeline.")

Exported 704 coefficients and the fitted pipeline.
